[← 05 - Subqueries to CTEs](<05-subqueries-to-ctes.ipynb>) · [Course Overview](<00-course-overview.ipynb>)

# 06 - Window Functions, Part 1: the group's answer, without losing the row

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/samietex/SQL_Tutorial/blob/main/06-window-functions-part-1.ipynb)

`05 - Subqueries to CTEs` ended with a report that told each customer whether they'd spent more than the average among customers who'd actually ordered something. Getting that one extra column meant a `customer_totals` CTE, a second `ordering_customers` CTE on top of it, and a scalar subquery inside the final `SELECT`: three moving parts to answer one question, how does this row compare to a number calculated across its whole group? Once you see the right tool, it takes one line.

*New here? The [Course Overview](<00-course-overview.ipynb>) explains how to run these notebooks. Viewing this on GitHub shows a preview with saved results; it won't execute anything.*

> **By the end of this notebook you'll be able to:** attach a group's total or average to every row with `OVER (PARTITION BY ...)`, rank rows within a group with `ROW_NUMBER()` and keep the top one, compare a row with the one before it using `LAG()`, and catch the two mistakes that either fail loudly or, worse, run and look right.

> **Read the full lesson:** this notebook is the hands-on companion to [Lesson 6 of the All About Data & More curriculum](LESSON_6_URL), which walks through the reasoning in more depth.

## 0. Set up

Run the cell below once each time you open the notebook. After it, any cell that starts with `%%sql` runs SQLite against a small lesson database that lives in memory, so there's nothing to install, and any cell that starts with `%%tsql` runs against SQL Server and AdventureWorks, which only section 10 uses and which has its own one-step setup there.

In [1]:
# Run this cell first. It downloads the small helper that lets the cells below run SQL,
# fetching a fresh copy every time so you always have the latest version.
import os, sys, tempfile, urllib.request
helper_dir = os.path.join(tempfile.gettempdir(), 'sql_tutorial_helper')
os.makedirs(helper_dir, exist_ok=True)
urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/samietex/SQL_Tutorial/assets/sql_tutorial.py',
    os.path.join(helper_dir, 'sql_tutorial.py'),
)
sys.path.insert(0, helper_dir)
%reload_ext sql_tutorial

Ready. %%sql runs SQLite 3.50.4 in memory, %%tsql runs SQL Server.


## 1. The same customers and orders

These are the four customers and five orders from `05`. Run the setup once; for a clean start later, run `%sql_reset` in a new cell and then run this cell again.

In [2]:
%%sql
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    customer_name TEXT NOT NULL,
    credit_limit INTEGER NOT NULL
);
CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL,
    amount INTEGER NOT NULL
);
INSERT INTO customers VALUES
    (1, 'Ada', 100),
    (2, 'Bola', 200),
    (3, 'Chidi', 300),
    (4, 'Dami', 200);
INSERT INTO orders VALUES
    (101, 1, 40),
    (102, 1, 60),
    (103, 2, 80),
    (104, 2, 90),
    (105, 3, 140);

4 rows inserted
5 rows inserted


## 2. What a window function actually does to the rows

`GROUP BY` answers "what's the total for each group" by collapsing every row in a group down to one summary row. That's exactly what you want sometimes, and exactly what you don't want other times, because once the rows are collapsed, the original detail is gone:

In [3]:
%%sql
SELECT
    customer_id,
    SUM(amount) AS customer_total
FROM orders
GROUP BY
    customer_id
ORDER BY
    customer_id;

customer_id,customer_total
1,100
2,170
3,140


Five orders went in and three rows came out, one per customer. Now ask a slightly different question: what if you still need to see each individual order, with that customer's total attached to it, so you can tell how much of the total any one order represents? A **window function** does exactly that:

In [4]:
%%sql
SELECT
    order_id,
    customer_id,
    amount,
    SUM(amount) OVER (PARTITION BY customer_id) AS customer_total
FROM orders
ORDER BY
    customer_id,
    order_id;

order_id,customer_id,amount,customer_total
101,1,40,100
102,1,60,100
103,2,80,170
104,2,90,170
105,3,140,140


This time five orders go in and five rows come back out. `PARTITION BY customer_id` inside `OVER (...)` says which rows count as one group for the calculation, the same idea as `GROUP BY`, but instead of collapsing the group into one row, the [SQLite window functions documentation](https://www.sqlite.org/windowfunctions.html) describes it as computing the value across that set of related rows and attaching it to every row in the set. Ada's two orders both show her full total of 100, and nothing about her individual order amounts got touched.

<p align="center">
  <img src="graphics/06_groupby_vs_window.png" width="560" alt="GROUP BY customer_id collapses five orders into three customer totals; SUM() OVER (PARTITION BY customer_id) keeps all five orders and attaches each customer total to them">
</p>

## 3. ROW_NUMBER(): ranking rows inside a group

The most common thing you'll do with a window function is rank the rows in a group against each other, and `ROW_NUMBER()` is the plain version: it numbers each row within its partition, starting at 1, in whatever order you tell it to use. Here's each customer's orders, ranked from largest:

In [5]:
%%sql
SELECT
    customer_id,
    order_id,
    amount,
    ROW_NUMBER() OVER (
        PARTITION BY customer_id
        ORDER BY amount DESC
    ) AS rn
FROM orders
ORDER BY
    customer_id,
    rn;

customer_id,order_id,amount,rn
1,102,60,1
1,101,40,2
2,104,90,1
2,103,80,2
3,105,140,1


Every order is still there, but each one now knows its rank inside its own customer's group. Ada's bigger order gets `rn = 1` and her smaller one `rn = 2`, and Bola's ranking starts again at 1 because `PARTITION BY` keeps the two customers' rankings apart. If all you want is each customer's single largest order, wrap the ranking in a derived table and filter on the rank, the "subquery as a table" move from `05`:

In [6]:
%%sql
SELECT
    customer_id,
    order_id,
    amount
FROM (
    SELECT
        customer_id,
        order_id,
        amount,
        ROW_NUMBER() OVER (
            PARTITION BY customer_id
            ORDER BY amount DESC
        ) AS rn
    FROM orders
) AS ranked
WHERE rn = 1
ORDER BY
    customer_id;

customer_id,order_id,amount
1,102,60
2,104,90
3,105,140


Hold onto why the filter had to move to an outer query instead of sitting in the same `SELECT` as `ROW_NUMBER()`. Section 8 comes back to it, and it's the kind of thing that errors loudly the first time you skip it, which is the friendlier way to fail.

## 4. LAG(): comparing a row to the one before it

`LAG()` looks back one row within the same partition, in whatever order you specify, and returns that row's value instead of the current row's. There's no join and no subquery, just a value pulled from the row that came before. Here's each customer's order compared to their own previous order:

In [7]:
%%sql
SELECT
    customer_id,
    order_id,
    amount,
    LAG(amount) OVER (PARTITION BY customer_id ORDER BY order_id) AS prev_order_amount,
    amount - LAG(amount) OVER (PARTITION BY customer_id ORDER BY order_id) AS change_from_prev
FROM orders
ORDER BY
    customer_id,
    order_id;

customer_id,order_id,amount,prev_order_amount,change_from_prev
1,101,40,NULL,NULL
1,102,60,40,20
2,103,80,NULL,NULL
2,104,90,80,10
3,105,140,NULL,NULL


Ada's second order jumped by 20 over her first, and Bola's by 10. Before window functions existed, this needed the orders table joined to itself on a condition like "the row whose order ID is one less than this one", which gets fragile as soon as order IDs aren't perfectly sequential for each customer. `LAG()` does it in a single pass, and `LEAD()` is the same idea looking forward instead of back.

## 5. Edge case: what LAG() returns when there's no previous row

Chidi only has one order, and both `prev_order_amount` and `change_from_prev` are `NULL` for him and for the first order of every other customer. That's `NULL`, not zero, and it matters which. There genuinely is no previous order for the first row in a partition, so `LAG()` can't return a real value, and `NULL` is SQL's honest way of saying there's nothing here rather than guessing at zero.

The part that catches people is what happens next: `amount - LAG(amount)` for that first row isn't `40 - 0`. Arithmetic with `NULL` produces `NULL`, so the whole expression is missing, not just half of it. If a report genuinely needs a zero there, say so with `COALESCE`, or give `LAG()` a default as its third argument:

In [8]:
%%sql
SELECT
    customer_id,
    order_id,
    amount,
    LAG(amount, 1, 0) OVER (PARTITION BY customer_id ORDER BY order_id) AS prev_or_zero,
    COALESCE(
        amount - LAG(amount) OVER (PARTITION BY customer_id ORDER BY order_id),
        0
    ) AS change_or_zero
FROM orders
ORDER BY
    customer_id,
    order_id;

customer_id,order_id,amount,prev_or_zero,change_or_zero
1,101,40,0,0
1,102,60,40,20
2,103,80,0,0
2,104,90,80,10
3,105,140,0,0


`LAG(amount, 1, 0)` means "one row back, and 0 if there isn't one". Notice the two choices aren't the same: with the default, Ada's first order shows a previous amount of 0, which claims she had a $0 order before, while `COALESCE` on the difference only says "no change to report". Pick the one that describes what the zero means in your report.

## 6. The callback: one CTE instead of two, plus a subquery

The last column of `05`'s report compared each customer's total against the average total among customers who'd placed at least one order, 136.67. Here's the same comparison with a window function, with the average rounded to two places to match:

In [9]:
%%sql
WITH customer_totals AS (
    SELECT
        c.customer_id,
        c.customer_name,
        c.credit_limit,
        COALESCE(SUM(o.amount), 0) AS total_spent
    FROM customers AS c
    LEFT JOIN orders AS o
        ON c.customer_id = o.customer_id
    GROUP BY
        c.customer_id,
        c.customer_name,
        c.credit_limit
)
SELECT
    customer_id,
    customer_name,
    total_spent,
    credit_limit,
    ROUND(AVG(total_spent) OVER (), 2) AS avg_spent,
    total_spent > AVG(total_spent) OVER () AS above_average_spender
FROM customer_totals
WHERE total_spent > 0
ORDER BY
    customer_id;

customer_id,customer_name,total_spent,credit_limit,avg_spent,above_average_spender
1,Ada,100,100,136.67,0
2,Bola,170,200,136.67,1
3,Chidi,140,300,136.67,1


The numbers match `05` exactly: Bola and Chidi above average, Ada below. The `WHERE total_spent > 0` still has to run before the average is computed, which keeps Dami's zero out of it, since `WHERE` runs before window functions do. What's gone is the second CTE: `AVG(total_spent) OVER ()`, with nothing inside the parentheses, means "compute this over every row left after `WHERE`", and it attaches the result to each of those rows instead of making you build a separate table to hold it.

## 7. Common mistake: forgetting PARTITION BY

Drop `PARTITION BY` from the `LAG()` example, and SQLite doesn't complain; it treats the entire result as one giant partition:

In [10]:
%%sql
SELECT
    customer_id,
    order_id,
    amount,
    LAG(amount) OVER (ORDER BY order_id) AS prev_amount
FROM orders
ORDER BY
    order_id;

customer_id,order_id,amount,prev_amount
1,101,40,NULL
1,102,60,40
2,103,80,60
2,104,90,80
3,105,140,90


Look at Bola's first order, 103. Its `prev_amount` is now 60, Ada's last order, because with no `PARTITION BY`, `order_id` alone decides what counts as "previous", and Ada's rows come right before Bola's in one long sequence. Nothing errors. The query returns a full result that looks entirely plausible to anyone who isn't already expecting `NULL` at the start of each customer's orders, which is why `PARTITION BY` deserves to be the first thing you check, not the last, when a `LAG()` or `LEAD()` result looks slightly off.

## 8. Common mistake: filtering a window alias in its own WHERE

Go back to the `ROW_NUMBER()` ranking and try to skip the derived table, filtering in the same `SELECT`. This cell is meant to fail:

In [11]:
%%sql --expect-error
SELECT
    customer_id,
    order_id,
    amount,
    ROW_NUMBER() OVER (
        PARTITION BY customer_id
        ORDER BY amount DESC
    ) AS rn
FROM orders
WHERE rn = 1;

Error (this cell is meant to fail): misuse of aliased window function rn


This isn't an accident of one engine. The [SQLite documentation states](https://www.sqlite.org/windowfunctions.html) that "window functions may only appear in the result set and in the ORDER BY clause of a SELECT statement", so `WHERE` is off the table by design. Window functions are calculated after `WHERE` has already chosen the rows, so when `WHERE` runs, `rn` doesn't exist yet:

<p align="center">
  <img src="graphics/06_where_before_window.png" width="720" alt="FROM, then WHERE, then GROUP BY and HAVING, then window functions, then the SELECT list, then ORDER BY; rn does not exist yet when WHERE runs">
</p>

Any time you need to act on a window function's result, whether that's keeping `rn = 1` or dropping rows below some cutoff, the pattern is the same: compute it in one query, and filter on it from the query one level up, either a derived table or a CTE.

## 9. The shape that stays the same

`ROW_NUMBER()`, `LAG()`, and `SUM() OVER ()` look like three different tools, but the shape around each of them never changes: `PARTITION BY` decides which rows belong together, `ORDER BY` decides what order they're considered in, and the function itself decides what to do with that group. Once that shape is automatic, the decision left in front of you is usually just which function goes in front of `OVER`.

<p align="center">
  <img src="graphics/06_window_overview.png" width="600" alt="One OVER (PARTITION BY ... ORDER BY ...) shape feeds three families: ranking with ROW_NUMBER, offset with LAG and LEAD, and aggregate with SUM and AVG OVER">
</p>

## 10. Scale it up: window functions in AdventureWorks

Five orders make every window checkable by hand, and real data is where window functions do the jobs that used to need self-joins and temporary tables. This section runs the lesson's patterns against **AdventureWorks2022**, Microsoft's sample database for an invented bike company, on SQL Server. Window functions work the same way there, so the only thing that changes is the scale.

### Set up AdventureWorks

SQL Server is a separate program from this notebook, so it needs one setup step before the `%%tsql` cells can run:

- **In Google Colab**, the cell below installs SQL Server and AdventureWorks inside your Colab session for you. The first run takes a few minutes and prints each step as it goes. Colab gives you a fresh machine every session, so run it again whenever you come back to the notebook later.
- **On your own machine**, it checks that SQL Server is reachable and tells you what's missing if it isn't; the [Course Overview](<00-course-overview.ipynb>) covers installing SQL Server and AdventureWorks locally.

If you'd rather not set anything up, every `%%tsql` cell below keeps its saved result, so you can read the section without running it.

In [12]:
# Run this once before any %%tsql cell.
# In Google Colab it installs SQL Server and AdventureWorks into this session (about 3 to 5
# minutes the first time, and again in each new session, because Colab starts fresh).
# On your own machine it just checks that your SQL Server is reachable.
%setup_adventureworks

AdventureWorks2022 is ready, so every %%tsql cell in this notebook will run.


### Each order's share of its customer's total

Customer 29825 placed twelve orders. `SUM() OVER (PARTITION BY ...)` puts their total on every row, so each order's share is one division away:

In [13]:
%%tsql
SELECT
    SalesOrderID,
    OrderDate,
    TotalDue,
    SUM(TotalDue) OVER (PARTITION BY CustomerID) AS customer_total,
    ROUND(TotalDue * 100.0 / SUM(TotalDue) OVER (PARTITION BY CustomerID), 2) AS percent_of_total
FROM Sales.SalesOrderHeader
WHERE CustomerID = 29825
ORDER BY
    OrderDate;

SalesOrderID,OrderDate,TotalDue,customer_total,percent_of_total
43659,2011-05-31 00:00:00,23153.2339,475892.0433,4.870000000000000
44305,2011-08-31 00:00:00,21262.6570,475892.0433,4.470000000000000
45061,2011-12-01 00:00:00,82490.9371,475892.0433,17.330000000000000
45779,2012-02-29 00:00:00,20304.9042,475892.0433,4.270000000000000
46604,2012-05-30 00:00:00,41824.8970,475892.0433,8.790000000000000
47693,2012-08-30 00:00:00,37295.9417,475892.0433,7.840000000000000
48730,2012-11-30 00:00:00,40510.6804,475892.0433,8.510000000000000
49822,2013-02-28 00:00:00,36282.6588,475892.0433,7.620000000000000
51081,2013-05-30 00:00:00,41328.4347,475892.0433,8.680000000000000
55234,2013-08-30 00:00:00,54244.2389,475892.0433,11.400000000000000


### The largest order in each territory

`ROW_NUMBER()` ranks every order within its sales territory, and the outer query keeps the top one. `SalesOrderID` is a second sort key inside `OVER`, so if two orders ever tied on `TotalDue`, the ranking would still be decided the same way every time:

In [14]:
%%tsql
SELECT
    TerritoryID,
    SalesOrderID,
    TotalDue
FROM (
    SELECT
        TerritoryID,
        SalesOrderID,
        TotalDue,
        ROW_NUMBER() OVER (
            PARTITION BY TerritoryID
            ORDER BY TotalDue DESC, SalesOrderID
        ) AS rn
    FROM Sales.SalesOrderHeader
) AS ranked
WHERE rn = 1
ORDER BY
    TerritoryID;

TerritoryID,SalesOrderID,TotalDue
1,51830,126852.1615
2,47395,165028.7482
3,51822,145454.3660
4,51131,187487.8250
5,44518,142312.2199
6,46616,170512.6689
7,46981,166537.0808
8,55254,117506.1173
9,53566,71729.8592
10,53573,130249.2557


Ten territories, ten rows, out of 31,465 orders, with no self-join and no `MAX` subquery per territory.

### Month over month with LAG()

This is the job `LAG()` is most often used for. A CTE adds up sales per month, and `LAG()` pulls in the month before, so each row can show its change:

In [15]:
%%tsql
WITH monthly_sales AS (
    SELECT
        DATEFROMPARTS(YEAR(OrderDate), MONTH(OrderDate), 1) AS month_start,
        SUM(TotalDue) AS sales
    FROM Sales.SalesOrderHeader
    GROUP BY
        DATEFROMPARTS(YEAR(OrderDate), MONTH(OrderDate), 1)
)
SELECT
    month_start,
    sales,
    LAG(sales) OVER (ORDER BY month_start) AS previous_month,
    sales - LAG(sales) OVER (ORDER BY month_start) AS change
FROM monthly_sales
WHERE month_start >= '2013-01-01'
  AND month_start < '2014-01-01'
ORDER BY
    month_start;

month_start,sales,previous_month,change
2013-01-01,2340061.5521,NULL,NULL
2013-02-01,2600218.8667,2340061.5521,260157.3146
2013-03-01,3831605.9389,2600218.8667,1231387.0722
2013-04-01,2840711.1734,3831605.9389,-990894.7655
2013-05-01,3658084.9461,2840711.1734,817373.7727
2013-06-01,5726265.2635,3658084.9461,2068180.3174
2013-07-01,5521840.8445,5726265.2635,-204424.4190
2013-08-01,3733973.0032,5521840.8445,-1787867.8413
2013-09-01,5083505.3374,3733973.0032,1349532.3342
2013-10-01,5374375.9418,5083505.3374,290870.6044


`DATEFROMPARTS` turns each order date into the first day of its month, so every order in a month lands in the same group. Look at January: its `previous_month` is `NULL`, even though the data has a December 2012. That's section 5 and section 8 meeting. `WHERE` keeps only the 2013 months before `LAG()` runs, so as far as `LAG()` is concerned, there is no month before January. To show January's change from December, compute `LAG()` over every month in a CTE first and filter to 2013 in the query above it.

There's no `PARTITION BY` here, deliberately: the question is about one continuous sequence of months, so the whole result is one window. Section 7's mistake is only a mistake when the rows belong to separate groups.

### The window alias in WHERE, on SQL Server

SQL Server applies the same rule as SQLite. This cell is meant to fail:

In [16]:
%%tsql --expect-error
SELECT
    SalesOrderID,
    TotalDue,
    ROW_NUMBER() OVER (ORDER BY TotalDue DESC) AS rn
FROM Sales.SalesOrderHeader
WHERE rn <= 5;

Error (this cell is meant to fail): Invalid column name 'rn'.


"Invalid column name 'rn'": the same reason, in different words. The fix is the same too: rank in an inner query and filter in the outer one.

## 11. Try it yourself

Write each query in a new cell before looking at the [solutions notebook](<solutions/06-window-functions-part-1-solutions.ipynb>), predicting the result first.

1. **Predict the running total.** Show each customer's running total, order by order, using `SUM(amount) OVER (PARTITION BY customer_id ORDER BY order_id)`. Before you run it, predict Ada's running total after her second order.
2. **Each order's share.** For every order, show what percentage of its customer's total it represents. Bola's two orders should add up to 100 percent.
3. **Each customer's latest order.** Return one row per customer: their most recent order (the highest `order_id`) and its amount.
4. **Stretch, on AdventureWorks.** For each salesperson, return their single largest order: `SalesPersonID`, `SalesOrderID`, and `TotalDue`. Leave out the orders with no salesperson.

## Before you trust a window function

- Check `PARTITION BY` first: does each window contain only the rows that belong together?
- Check `ORDER BY` inside `OVER`: for ranking and `LAG()`, can two rows tie? If so, add a tie-breaker.
- The first row of each partition gets `NULL` from `LAG()`; decide whether that should stay `NULL` or become a zero.
- To filter on a window function's result, compute it in an inner query and filter in the outer one.
- Remember that `WHERE` runs first, so it also decides which rows the window can see.

## Go deeper

- **[Window Functions](https://www.sqlite.org/windowfunctions.html)**, SQLite. The full reference, including every built-in window function and the frame options `07` covers.
- **[PostgreSQL tutorial: Window Functions](https://www.postgresql.org/docs/current/tutorial-window.html)**. A short, clear introduction with the same "aggregate without collapsing" framing.
- **[OVER clause](https://learn.microsoft.com/en-us/sql/t-sql/queries/select-over-clause-transact-sql)**, **[ROW_NUMBER](https://learn.microsoft.com/en-us/sql/t-sql/functions/row-number-transact-sql)**, and **[LAG](https://learn.microsoft.com/en-us/sql/t-sql/functions/lag-transact-sql)**, Microsoft Learn. The SQL Server references behind section 10.
- **[Window functions](https://use-the-index-luke.com/sql/partial-results/window-functions)**, Markus Winand. How window functions replace slower self-joins and subqueries, from *Use The Index, Luke*.
- **Practice:** [SQLZoo: Window functions](https://sqlzoo.net/wiki/Window_functions) and [DataLemur's window functions tutorial](https://datalemur.com/sql-tutorial/sql-aggregate-window-functions), which covers the ranking questions interviews love.

For interview questions on this topic with model answers, see the [study guide](<study-guides/06-window-functions-part-1.md>).

## What's next

The running-total exercise leaves a question open: when you add `ORDER BY` inside `OVER`, exactly which rows does the sum include, and what happens when two rows tie on the `ORDER BY` column? `07 - Window Functions, Part 2` names that hidden default, the window frame, and explains a `LAST_VALUE()` result that looks wrong until you see why.

---

[← 05 - Subqueries to CTEs](<05-subqueries-to-ctes.ipynb>) · [Course Overview](<00-course-overview.ipynb>) · [07 - Window Functions, Part 2 →](<07-window-functions-part-2.ipynb>)

*SQL_Tutorial* is written and maintained by Samuel Shaibu as part of *All About Data & More*. Licensed under [MIT](LICENSE).